Python code implementing VAR-LiNGAM of nominal exchange rates and relative prices. 
These scripts replicate results included in Table 7 (pre-zero matrix B0)   of "Identifying the Causal Relationship between Exchange Rates and Prices" by Hironobu Nakagawa.

The LiNGAM algorithms used in the present analysis is developed by Shimizu (2006) and Hyvärinen et al.(2010).
References:
S. Shimizu, P. O. Hoyer, A. Hyvärinen, and A. J. Kerminen.  A linear non-gaussian acyclic model for causal discovery.  Journal of Machine Learning Research, 7: 2003-2030, 2006.
A. Hyvärinen, K. Zhang, S. Shimizu, and P. O. Hoyer. Estimation of a structural vector autoregression model using non-Gaussianity. Journal of Machine Learning Research, 11: 1709-1731, 2010.

In [25]:
import pandas as pd
import numpy as np
from statsmodels.tsa.vector_ar.var_model import VAR
from sklearn.decomposition import FastICA
from scipy.optimize import linear_sum_assignment
from sklearn.linear_model import LinearRegression
import warnings
warnings.simplefilter('ignore')
np.set_printoptions(precision=5, suppress=True)

In [26]:
#load data; choose a sheet
rawdf = pd.read_excel("ER-P-mData.xlsx",sheet_name='Data')
df = rawdf.copy()

In [27]:
#  Analysis function
def analyze_country_pair(df, sn_var, pn_var):

    # Transform raw variables
    df['ER'] = np.log(1 / df[sn_var])
    df['pforeign'] = np.log(df[pn_var])
    df['phome'] = np.log(df['pus'])
    df['𝜋'] = df['pforeign'] - df['phome']
    df['rer'] = df['ER'] + df['𝜋']
    df['drer'] = df['rer'] - df['rer'].shift(1)
      
    # Differencing (applicable for models using differenced variables) and standardization
    df['dER'] = df['ER'].diff()
    df['d𝜋'] = df['𝜋'].diff()

    # Demean series (level and index series)
    df['ER'] = (df['ER'] - df['ER'].mean()) 
    df['𝜋'] = (df['𝜋'] - df['𝜋'].mean()) 
    
    # Choose level or differenced series
    #X = df[['ER', '𝜋']].dropna()
    X = df[['dER', 'd𝜋']].dropna()

    # Estimate VAR (with "aic" or "bic") using data X and obtain residuals
    var = VAR(X)
    result = var.fit(maxlags=4, ic="bic", trend="n")
    resid_VAR = result.resid
        
    # 1) run FastICA
    ica = FastICA(random_state=1, max_iter=10000)
    ica.fit(resid_VAR)
    #ica.fit(resid_df_1[['ER','𝜋']])

    W_ica = ica.components_   # unmixing matrix
    # 2) permute W_ica using linear_sum_assignment exactly as in code
    _, col_index = linear_sum_assignment(1 / np.abs(W_ica))
    PW_ica = np.zeros_like(W_ica)
    PW_ica[col_index] = W_ica
    # 3) scale and form W_estimate and B_estimate (this is the pre-zero matrix)
    D = np.diag(PW_ica)[:, np.newaxis]   # column vector of diagonal
    W_estimate = PW_ica / D
    B_estimate = np.eye(W_estimate.shape[0]) - W_estimate

    #Inspect the pre-zeroed matrix
    #print("W_ica:\n", W_ica)
    #print("W_esta (permuted):\n", W_estimate)
    print(f'{sn_var}-{pn_var}')
    print("pre-zeroed B_est:\n", B_estimate)
    print('------')

    return {
        'pair': f'{sn_var}-{pn_var}',
        'pre-zeroed B_est': B_estimate,
        
    }


In [28]:
country_pairs = [
    ('sca', 'pca'),
    ('sden', 'pden'),
    ('seuro', 'peuro'),
    ('sja', 'pja'),
    ('snor', 'pnor'),
    ('sswe', 'pswe'),
    ('sswi', 'pswi'),
    ('suk', 'puk')
]

print('Pre-zero matrix B0, full sample')
summary_table = []

for sn_var, pn_var in country_pairs:
    result = analyze_country_pair(df, sn_var, pn_var)

    summary_table.append({
        'Pair': result['pair'],
        'Pre-zeroed B_est': result['pre-zeroed B_est'],
        
    })

Pre-zero matrix B0, full sample
sca-pca
pre-zeroed B_est:
 [[ 0.       0.14795]
 [-0.01425  0.     ]]
------
sden-pden
pre-zeroed B_est:
 [[ 0.      -0.02883]
 [-0.01525  0.     ]]
------
seuro-peuro
pre-zeroed B_est:
 [[ 0.      -0.47263]
 [ 0.00737  0.     ]]
------
sja-pja
pre-zeroed B_est:
 [[ 0.      -0.14849]
 [ 0.02023  0.     ]]
------
snor-pnor
pre-zeroed B_est:
 [[ 0.       0.08968]
 [-0.02264  0.     ]]
------
sswe-pswe
pre-zeroed B_est:
 [[ 0.       0.16795]
 [-0.01747  0.     ]]
------
sswi-pswi
pre-zeroed B_est:
 [[ 0.      -0.24178]
 [-0.00681  0.     ]]
------
suk-puk
pre-zeroed B_est:
 [[ 0.       0.2743 ]
 [-0.01375  0.     ]]
------


In [29]:
summary_df = pd.DataFrame(summary_table)
pd.set_option('display.max_columns', None)
print("\n📋 Summary Table Across Country Pairs:")
print(summary_df)


📋 Summary Table Across Country Pairs:
          Pair                                   Pre-zeroed B_est
0      sca-pca  [[0.0, 0.14795365205811523], [-0.0142497339168...
1    sden-pden  [[0.0, -0.02883077177902378], [-0.015254548561...
2  seuro-peuro  [[0.0, -0.472628134500585], [0.007370985218652...
3      sja-pja  [[0.0, -0.14849459894065387], [0.0202322555671...
4    snor-pnor  [[0.0, 0.08968301907134009], [-0.0226370962800...
5    sswe-pswe  [[0.0, 0.16794943308441493], [-0.0174704096037...
6    sswi-pswi  [[0.0, -0.2417813637166051], [-0.0068056606230...
7      suk-puk  [[0.0, 0.27429859864756245], [-0.0137521658756...


In [39]:
# Now, obtain results using vecm and mean-reverting sample
#  Analysis function
def analyzeMR2_country_pair(df, sn_var, pn_var):

    # Transform raw variables
    df['ER'] = np.log(1 / df[sn_var])
    df['pforeign'] = np.log(df[pn_var])
    df['phome'] = np.log(df['pus'])
    df['𝜋'] = df['pforeign'] - df['phome']
    df['rer'] = df['ER'] + df['𝜋']
    df['drer'] = df['rer'] - df['rer'].shift(1)
      
    # Differencing (applicable for models using differenced variables) and standardization
    df['dER'] = df['ER'].diff()
    df['d𝜋'] = df['𝜋'].diff()

           
    #----------------
    # Two-regime model ( mean reverting sample belongs to regime 1)
    #-----------------
    # create index: 1 for mean reversion, 0 otherwise
    df['index'] = np.where(
    df['drer'] *  (df['rer'] - df['rer'].mean())< 0,
    1,
    0
    )
    # Create the error correction term (if VECM is applicable)
    df['ECT_lag1'] =  df['ER'].shift(1) + df['𝜋'].shift(1)
       
    # choose vaiables in first difference
    df['ER'] = df['ER'].diff()
    df['𝜋'] = df['𝜋'].diff()

    df = df[['ER', '𝜋','ECT_lag1', 'index']].copy()

    # Create lagged RHS variables from full sample
    p = 1
    for lag in range(1, p + 1):
        df[f'ER_lag{lag}'] = df['ER'].shift(lag)
        df[f'𝜋_lag{lag}'] = df['𝜋'].shift(lag)

    # Drop initial rows with NaNs due to lagging
    df_lagged = df.dropna().copy()

    # Define RHS predictors including the ECT if needed
    rhs_cols = (
    [f'ER_lag{lag}' for lag in range(1, p + 1)] +
    [f'𝜋_lag{lag}' for lag in range(1, p + 1)] + ['ECT_lag1']
    )
    # Split data by regime
    lhs_1 = df_lagged[df_lagged['index'] == 1]
    lhs_0 = df_lagged[df_lagged['index'] == 0]

    X_1 = lhs_1[rhs_cols].values
    X_0 = lhs_0[rhs_cols].values

    residuals_1 = {}
    residuals_0 = {}

    import statsmodels.api as sm
    # Loop over target variables and fit separate models
    for target in ['ER', '𝜋']:
        y_1 = lhs_1[target].values
        y_0 = lhs_0[target].values

        # Fit model within regime 1
        residuals_1[target] = sm.OLS(y_1,X_1).fit().resid
        
        # Fit model within regime 0
        residuals_0[target] = sm.OLS(y_0,X_0).fit().resid

    # Organize into DataFrames
    resid_df_1 = pd.DataFrame(residuals_1, index=lhs_1.index)
    resid_df_0 = pd.DataFrame(residuals_0, index=lhs_0.index)

        
    # 1) run FastICA
    icaMR2 = FastICA(random_state=1, max_iter=10000)
    icaMR2.fit(resid_df_1[['ER', '𝜋']])
    
    W_icaMR2 = icaMR2.components_   # unmixing matrix
    # 2) permute W_ica using linear_sum_assignment exactly as in code
    _, col_index = linear_sum_assignment(1 / np.abs(W_icaMR2))
    PW_icaMR2 = np.zeros_like(W_icaMR2)
    PW_icaMR2[col_index] = W_icaMR2
    # 3) scale and form W_estimate and B_estimate (this is the pre-zero matrix)
    Dmr2 = np.diag(PW_icaMR2)[:, np.newaxis]   # column vector of diagonal
    W_estimateMR2 = PW_icaMR2 / Dmr2
    B_estimateMR2 = np.eye(W_estimateMR2.shape[0]) - W_estimateMR2

    #Inspect the pre-zeroed matrix
    #print("W_icaMR2:\n", W_icaMR2)
    #print("W_estaMR2 (permuted):\n", W_estimateMR2)
    print(f'{sn_var}-{pn_var}')
    print("pre-zeroed B_est (MR2):\n", B_estimateMR2)
    print('------')

    return {
        'pair': f'{sn_var}-{pn_var}',
        'pre-zeroed B_estMR2': B_estimateMR2,
        
    }



In [40]:
country_pairs = [
    ('sca', 'pca'),
    ('sden', 'pden'),
    ('seuro', 'peuro'),
    ('sja', 'pja'),
    ('snor', 'pnor'),
    ('sswe', 'pswe'),
    ('sswi', 'pswi'),
    ('suk', 'puk')
]

print('Pre-zero matrix B0, vecm, Mean-reverting sample ')
summary_table = []

for sn_var, pn_var in country_pairs:
    result = analyzeMR2_country_pair(df, sn_var, pn_var)

    summary_table.append({
        'Pair': result['pair'],
        'Pre-zeroed B_estMR2': result['pre-zeroed B_estMR2'],
        
    })

Pre-zero matrix B0, vecm, Mean-reverting sample 
sca-pca
pre-zeroed B_est (MR2):
 [[ 0.      -0.13252]
 [-0.04724  0.     ]]
------
sden-pden
pre-zeroed B_est (MR2):
 [[ 0.       0.18228]
 [-0.0233   0.     ]]
------
seuro-peuro
pre-zeroed B_est (MR2):
 [[ 0.      -0.54111]
 [-0.06883  0.     ]]
------
sja-pja
pre-zeroed B_est (MR2):
 [[0.      0.16872]
 [0.02485 0.     ]]
------
snor-pnor
pre-zeroed B_est (MR2):
 [[ 0.       1.76246]
 [-0.08415  0.     ]]
------
sswe-pswe
pre-zeroed B_est (MR2):
 [[ 0.      -0.42513]
 [ 0.01863  0.     ]]
------
sswi-pswi
pre-zeroed B_est (MR2):
 [[ 0.      -1.25746]
 [ 0.01728  0.     ]]
------
suk-puk
pre-zeroed B_est (MR2):
 [[0.      0.13316]
 [0.00171 0.     ]]
------


In [41]:
summary_dfMR2 = pd.DataFrame(summary_table)
pd.set_option('display.max_columns', None)
print("\n📋 Summary Table Across Country Pairs:")
print(summary_dfMR2)


📋 Summary Table Across Country Pairs:
          Pair                                Pre-zeroed B_estMR2
0      sca-pca  [[0.0, -0.13251741706479744], [-0.047242270850...
1    sden-pden  [[0.0, 0.1822806041656629], [-0.02329818055949...
2  seuro-peuro  [[0.0, -0.5411147415329446], [-0.0688291529226...
3      sja-pja  [[0.0, 0.1687212659191047], [0.024849169769151...
4    snor-pnor  [[0.0, 1.7624607332432476], [-0.08415464765367...
5    sswe-pswe  [[0.0, -0.42512898895352424], [0.0186273134490...
6    sswi-pswi  [[0.0, -1.2574611485530132], [0.01727707259634...
7      suk-puk  [[0.0, 0.13316274119080548], [0.00171166524619...
